In [5]:
import pandas as pd 
import numpy as np 

#ML
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

In [6]:
network_list = [
    'conf','email_eu','hospital','school','work'
]

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from scipy import stats
import matplotlib.pyplot as plt
import os

def confidence_interval(accuracy, n, confidence=0.95):
    std_error = np.sqrt(accuracy * (1 - accuracy) / n)
    z_score = stats.norm.ppf((1 + confidence) / 2)
    margin_of_error = z_score * std_error
    return (accuracy - margin_of_error, accuracy + margin_of_error)

# Assume network_list is defined

for network_name in network_list:
    # Read data for the current network
    complex_c = pd.read_csv(f'../results/unweighted/given_q_n=300/unweighted_complex_{network_name}_EPH.csv')
    simple_c = pd.read_csv(f'../results/unweighted/given_q_n=300/unweighted_simple_{network_name}_EPH.csv')

    # Lists to store results
    q_set = [-1,0.1,0.2,0.3,0.4,0.5]
    accuracies_tda = []
    ci_lower_tda = []
    ci_upper_tda = []
    accuracies_prl = []
    ci_lower_prl = []
    ci_upper_prl = []

    for q in q_set:
        # Prepare data
        complex_curr = complex_c[complex_c['q'] == q]
        complex_curr = complex_curr[['EPH','corr']]
        complex_curr['complex'] = 1
        
        simple_curr = simple_c[['EPH','corr']]
        simple_curr['complex'] = 0
        
        merged = pd.concat([simple_curr,complex_curr])
        merged = merged.sample(len(merged),random_state=1)
        
        X_tda = merged['EPH']
        X_prl = merged['corr']
        y = merged['complex']

        # Split data
        X_tda_train, X_tda_test, y_tda_train, y_tda_test = train_test_split(X_tda, y, test_size=0.3, random_state=42)
        X_prl_train, X_prl_test, y_prl_train, y_prl_test = train_test_split(X_prl, y, test_size=0.3, random_state=42)

        # Train and evaluate models
        dt_tda = DecisionTreeClassifier(random_state=42)
        dt_prl = DecisionTreeClassifier(random_state=42)

        dt_tda.fit(X_tda_train.values.reshape(-1, 1), y_tda_train)
        dt_prl.fit(X_prl_train.values.reshape(-1, 1), y_prl_train)

        y_tda_pred = dt_tda.predict(X_tda_test.values.reshape(-1, 1))
        y_prl_pred = dt_prl.predict(X_prl_test.values.reshape(-1, 1))

        accuracy_tda = accuracy_score(y_tda_test, y_tda_pred)
        accuracy_prl = accuracy_score(y_prl_test, y_prl_pred)

        # Calculate confidence intervals
        ci_tda = confidence_interval(accuracy_tda, len(y_tda_test))
        ci_prl = confidence_interval(accuracy_prl, len(y_prl_test))

        # Store results
        accuracies_tda.append(accuracy_tda)
        ci_lower_tda.append(ci_tda[0])
        ci_upper_tda.append(ci_tda[1])
        accuracies_prl.append(accuracy_prl)
        ci_lower_prl.append(ci_prl[0])
        ci_upper_prl.append(ci_prl[1])

    # Create the plot
    plt.figure(figsize=(8, 6))

    plt.plot([0,0.1,0.2,0.3,0.4,0.5], accuracies_tda, marker='o', label='EPH')
    plt.fill_between([0,0.1,0.2,0.3,0.4,0.5], ci_lower_tda, ci_upper_tda, alpha=0.2)

    plt.plot([0,0.1,0.2,0.3,0.4,0.5], accuracies_prl, marker='s', label='corr(order,deg)')
    plt.fill_between([0,0.1,0.2,0.3,0.4,0.5], ci_lower_prl, ci_upper_prl, alpha=0.2)
    
    # Set x-axis to show only integer values
    plt.xticks([0,0.1,0.2,0.3,0.4,0.5])


    plt.xlabel('q',fontsize=15)
    plt.ylabel('Accuracy',fontsize=15)
    plt.legend(loc='lower right',fontsize=15)
    
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    
    plt.grid(True)

    plt.tight_layout()
    
    # Save the plot
    plt.savefig(f'../results/unweighted_plots/n=300_given_q/{network_name}_accuracy_plot.pdf', format='pdf', dpi=1200, bbox_inches='tight')
    plt.close()  # Close the plot to free up memory


/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_6660/3726489687.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  simple_curr['complex'] = 0
/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_6660/3726489687.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  simple_curr['complex'] = 0
/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_6660/3726489687.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,c